# 17.5 Q-Learning

**中文**：上一节的 SARSA 是**同策略(on-policy)**——它学习的是"我实际执行的(带探索的)策略"的价值。本节的 **Q-Learning(Watkins, 1989)** 是**异策略(off-policy)** 的、也是**最著名、最重要的经典 RL 算法**:它一边用 ε-贪心到处探索(行为策略)，一边学习"**假设永远贪心的最优策略**"的价值(目标策略)。两者可以不同——这就是"异策略"。Q-learning 简单、强大、有收敛保证，是通往深度强化学习(DQN)的直接跳板。
**English**: SARSA (last section) is **on-policy** — it learns the value of "the (exploration-including) policy I actually follow." This section's **Q-Learning (Watkins, 1989)** is **off-policy** and the **most famous, most important classic RL algorithm**: it explores with ε-greedy (the behavior policy) while learning the value of the "**always-greedy optimal policy**" (the target policy). The two can differ — that is "off-policy." Q-learning is simple, powerful, has convergence guarantees, and is the direct springboard to deep RL (DQN).

---

**中文**：**Q-learning 更新公式**——和 SARSA 只差一个字:
**English**: The **Q-learning update** — differing from SARSA by just one word:

$$Q(s,a)\leftarrow Q(s,a)+\alpha\Big[\,r+\gamma\,\underbrace{\max_{a'}Q(s',a')}_{\text{目标:贪心最优}}-Q(s,a)\Big]$$

**中文**：关键就在那个 **$\max_{a'}$**:
**English**: The key is that **$\max_{a'}$**:
- **SARSA** 用 $Q(s',a')$——$a'$ 是**实际会执行的下一个动作**(含 ε 探索)。学"我这个爱探索的策略"的价值。
  **SARSA** uses $Q(s',a')$ — $a'$ is the **action actually taken next** (with ε-exploration). Learns the value of "my exploring policy."
- **Q-learning** 用 $\max_{a'}Q(s',a')$——**不管下一步实际做什么，都假设下一步会选最优动作**。学"最优贪心策略"的价值，与行为策略脱钩(off-policy)。
  **Q-learning** uses $\max_{a'}Q(s',a')$ — **regardless of what is actually done next, it assumes the best action next**. Learns the "optimal greedy policy" value, decoupled from the behavior policy (off-policy).

**中文**：这一个 max 带来深远影响:Q-learning **收敛到最优 $Q^*$**(只要所有状态-动作对被无限次访问)，即使数据来自随机探索、甚至来自别的策略。这正是 **DQN 用经验回放池"复用旧数据"** 的理论前提。
**English**: This single max has deep consequences: Q-learning **converges to the optimal $Q^*$** (given all state-action pairs are visited infinitely often), even if the data comes from random exploration or another policy. This is exactly the theoretical basis for **DQN reusing old data from a replay buffer**.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 最高频)**
> **中文**：**Q-learning=off-policy TD 控制**, 更新用 $\max_{a'}Q(s',a')$(SARSA 用实际 $a'$)。**off-policy 三大好处**:①能学最优策略同时任意探索; ②能**复用历史/他人数据**(→经验回放→DQN); ③行为与目标策略解耦。收敛到 $Q^*$(需充分探索+合适 α)。**经典陷阱**:因为 max 有**过估计偏差(overestimation)**→ Double Q-learning 修正。**SARSA vs Q-learning(悬崖行走)**:Q-learning 学到**最优但贴崖的危险路**、训练时在线回报更差(探索会掉崖); SARSA 学**安全绕行路**、在线回报更好——这是"你评估谁的策略"的直接体现。
> **English**: **Q-learning = off-policy TD control**, updating with $\max_{a'}Q(s',a')$ (SARSA uses the actual $a'$). **Three benefits of off-policy**: ① learn the optimal policy while exploring arbitrarily; ② **reuse historical/others' data** (→ experience replay → DQN); ③ decouple behavior from target policy. Converges to $Q^*$ (with enough exploration + suitable α). **Classic pitfall**: the max causes **overestimation bias** → fixed by Double Q-learning. **SARSA vs Q-learning (Cliff Walking)**: Q-learning learns the **optimal but cliff-hugging risky path** with worse online reward during training (exploration falls off); SARSA learns the **safe detour** with better online reward — a direct consequence of "whose policy you evaluate."


In [ ]:

# ============================================================
# 环境:从零实现 Taxi / Taxi from scratch (500 states, 6 actions)
# 中文:5x5 网格接送乘客。4 个标志点 R/G/Y/B。出租车要:开到乘客处→接客→开到目的地→放客。
#      状态=出租车位置(25)×乘客位置(5:4个点或车上)×目的地(4)=500。动作:南北东西+接客+放客。
#      每步 -1; 成功放客 +20; 非法接/放客 -10。有内墙挡住部分东西向移动。
# English: 5x5 grid taxi. 4 landmarks R/G/Y/B. The taxi must: drive to passenger → pick up →
#      drive to destination → drop off. State = taxi pos(25) × passenger(5) × destination(4) = 500.
#      Actions: N/S/E/W + pickup + dropoff. Step -1; successful dropoff +20; illegal pick/drop -10.
# ============================================================
import numpy as np, matplotlib.pyplot as plt
rng=np.random.default_rng(0)
LOC=[(0,0),(0,4),(4,0),(4,3)]                                # R,G,Y,B 的坐标 / landmark coords
WALLS=set()                                                   # 内墙(挡东西移动)/ interior walls
for r in [0,1]: WALLS.add(((r,1),(r,2)))
for r in [3,4]: WALLS.add(((r,0),(r,1))); WALLS.add(((r,2),(r,3)))
def blocked(a,b): return (a,b) in WALLS or (b,a) in WALLS
def encode(tr,tc,pl,dst): return ((tr*5+tc)*5+pl)*4+dst      # 状态编码 / encode to 0..499
def decode(s):
    dst=s%4; s//=4; pl=s%5; s//=5; tc=s%5; tr=s//5; return tr,tc,pl,dst
nS,nA=500,6
def step(s,a):
    tr,tc,pl,dst=decode(s); r=-1.0; done=False; ntr,ntc=tr,tc
    if   a==0: ntr=min(tr+1,4)                                # 南 / south
    elif a==1: ntr=max(tr-1,0)                                # 北 / north
    elif a==2: ntc=tc+1 if tc<4 and not blocked((tr,tc),(tr,tc+1)) else tc   # 东 / east (墙则不动)
    elif a==3: ntc=tc-1 if tc>0 and not blocked((tr,tc),(tr,tc-1)) else tc   # 西 / west
    elif a==4:                                                # 接客 / pickup
        if pl<4 and (tr,tc)==LOC[pl]: pl=4
        else: r=-10.0
    elif a==5:                                                # 放客 / dropoff
        if pl==4 and (tr,tc)==LOC[dst]: r=20.0; done=True
        else: r=-10.0
    return encode(ntr,ntc,pl,dst), r, done
def reset():
    while True:
        tr,tc,pl,dst=rng.integers(5),rng.integers(5),rng.integers(4),rng.integers(4)
        if pl!=dst: return encode(tr,tc,pl,dst)
print("状态数 / #states:", nS, "| 动作数 / #actions:", nA, "(南北东西+接客+放客)")


**中文**：跑 **Q-learning**:ε-贪心探索(ε 随训练衰减)，每步用 $\max$ 目标更新 $Q$。看它能否学会"接客→送达"这条需要**多步规划**的任务。
**English**: Run **Q-learning**: ε-greedy exploration (ε decays over training), updating $Q$ with the $\max$ target each step. Can it learn the multi-step "pick up → deliver" task?


In [ ]:

# ============================================================
# Q-learning 训练 + 评估 / Q-learning train & evaluate
# ============================================================
def q_learning(episodes=15000, alpha=0.5, gamma=0.99):
    Q=np.zeros((nS,nA)); ep_rewards=[]
    for ep in range(episodes):
        s=reset(); tot=0; eps=max(0.02, 0.1*(1-ep/episodes))  # ε 线性衰减 / decaying exploration
        for _ in range(200):
            a=int(rng.integers(nA)) if rng.random()<eps else int(np.argmax(Q[s]))  # ε-贪心行为 / behavior
            ns,r,done=step(s,a); tot+=r
            Q[s,a]+=alpha*(r + gamma*np.max(Q[ns])*(not done) - Q[s,a])   # off-policy: 用 max / max target
            s=ns
            if done: break
        ep_rewards.append(tot)
    return Q, ep_rewards

Q, rewards = q_learning(15000)
# 贪心评估 / greedy evaluation
def evaluate(Q, n=2000):
    total=0; success=0; steps=0
    for _ in range(n):
        s=reset()
        for t in range(200):
            s,r,done=step(s,int(np.argmax(Q[s]))); total+=r; steps+=1
            if done: success+=1; break
    return total/n, success/n, steps/n
avg,succ,st=evaluate(Q)
print(f"贪心策略:平均回报 {avg:.2f}, 成功率 {succ:.1%}, 平均步数 {st:.1f}")
print(f"Greedy policy: avg return {avg:.2f}, success rate {succ:.1%}, avg steps {st:.1f}")
print("→ Q-learning 学会了多步规划:先去接客再送到目的地 / learned to pick up then deliver")


**中文**：现在是 RL 最经典的教学对比——**在悬崖行走上让 SARSA 和 Q-learning 正面对决**。同样的环境、同样的 ε-贪心探索，只因更新公式里"用实际 $a'$"还是"用 $\max$"的一字之差，两者学出**截然不同的策略**。
**English**: Now RL's most classic teaching comparison — **SARSA vs Q-learning head-to-head on Cliff Walking**. Same environment, same ε-greedy exploration; only the one-word difference ("actual $a'$" vs "$\max$") in the update makes them learn **completely different policies**.


In [ ]:

# ============================================================
# 悬崖行走:SARSA vs Q-learning / Cliff Walking showdown
# ============================================================
H,W=4,12; ACT=[(-1,0),(0,1),(1,0),(0,-1)]; cstart=3*W+0; cgoal=3*W+11; ccliff=set(3*W+c for c in range(1,11))
def cstep(s,a):
    r,c=divmod(s,W); dr,dc=ACT[a]; nr,nc=max(0,min(H-1,r+dr)),max(0,min(W-1,c+dc)); ns=nr*W+nc
    if ns in ccliff: return cstart,-100.0,False
    return ns,-1.0,(ns==cgoal)
def cliff_run(mode, episodes=500, alpha=0.5, eps=0.1, gamma=1.0):
    Q=np.zeros((W*H,4)); rews=[]
    act=lambda s: int(rng.integers(4)) if rng.random()<eps else int(np.argmax(Q[s]))
    for _ in range(episodes):
        s=cstart; a=act(s); tot=0
        for _ in range(200):
            ns,r,d=cstep(s,a); tot+=r
            if mode=="SARSA":
                na=act(ns); Q[s,a]+=alpha*(r+gamma*Q[ns,na]*(not d)-Q[s,a]); s,a=ns,na   # 用实际 a'
            else:
                Q[s,a]+=alpha*(r+gamma*np.max(Q[ns])*(not d)-Q[s,a]); s=ns; a=act(s)      # 用 max
            if d: break
        rews.append(tot)
    return Q, rews
def greedy_path(Q):
    s=cstart; p=[s]
    for _ in range(40):
        s=cstep(s,int(np.argmax(Q[s])))[0]; p.append(s)
        if s==cgoal: break
    return p
Qs,rs=cliff_run("SARSA"); Qq,rq=cliff_run("Qlearning")
ps,pq=greedy_path(Qs), greedy_path(Qq)
print(f"SARSA     : 贪心路 {len(ps)-1} 步 (安全绕行), 训练在线回报均值 {np.mean(rs[-100:]):.1f}")
print(f"Q-learning: 贪心路 {len(pq)-1} 步 (贴崖最优), 训练在线回报均值 {np.mean(rq[-100:]):.1f}")
print("诚实反直觉:Q-learning 学到更优的路, 但训练时在线回报却更差(探索会掉崖)!")


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,3,figsize=(17,4.6))
# ① Taxi 学习曲线 / Taxi learning curve
rw=np.array(rewards); sm=np.convolve(rw,np.ones(100)/100,mode="valid")
ax[0].plot(rw,alpha=0.2,color="#4C72B0"); ax[0].plot(sm,color="#C44E52",lw=2)
ax[0].axhline(avg,ls="--",color="gray",label=f"greedy {avg:.1f}")
ax[0].set_title("Taxi:Q-learning 学习曲线 / learning curve"); ax[0].set_xlabel("episode"); ax[0].set_ylabel("return"); ax[0].set_ylim(-400,30); ax[0].legend()
# ② 悬崖两条路 / the two cliff paths
def draw_cliff(a,path,title,color):
    board=np.zeros((H,W));
    for c in range(1,11): board[3,c]=-1
    a.imshow(board,cmap="Reds_r",vmin=-1,vmax=1)
    a.text(0,3,"S",ha="center",va="center"); a.text(11,3,"G",ha="center",va="center")
    for c in range(1,11): a.text(c,3,"✗",ha="center",va="center",fontsize=8)
    pr=[divmod(s,W) for s in path]; a.plot([c for r,c in pr],[r for r,c in pr],"o-",color=color,lw=2,ms=4)
    a.set_title(title); a.set_xticks([]); a.set_yticks([])
draw_cliff(ax[1],ps,f"SARSA:安全绕行({len(ps)-1}步)/ safe","#4C72B0")
draw_cliff(ax[2],pq,f"Q-learning:贴崖最优({len(pq)-1}步)/ optimal risky","#55A868")
plt.tight_layout(); plt.savefig("/tmp/rl05_viz.png",dpi=80); plt.show()
# 在线回报对比 / online reward comparison
print(f"训练在线平均回报:SARSA {np.mean(rs[-100:]):.1f}  >  Q-learning {np.mean(rq[-100:]):.1f}")
print("但贪心(最终)策略:Q-learning 更短更优 / but Q-learning's greedy policy is shorter/optimal")


**中文**：诚实解读:
**English**: Honest takeaways:

**中文**：
1. **Q-learning 解决了 Taxi** —— 一个需要"先接客、再送达"的多步规划任务，贪心策略成功率 100%、平均回报接近最优(~+7.9)。它从零(全 0 的 Q 表)、只靠 −1/+20/−10 的稀疏奖励和试错，学会了整套接送流程。
2. **悬崖对决揭示 off-policy 的本质**:同样的探索，Q-learning 学出**贴着悬崖的最短路(13步,最优)**，SARSA 学出**远离悬崖的安全路(17步)**。因为 Q-learning 用 $\max$——它评估的是"如果我以后都贪心(不探索)会怎样",所以敢走崖边;SARSA 评估的是"我这个会 10% 乱走的策略会怎样",走崖边迟早掉下去,于是绕行。
3. **反直觉的诚实结果**:虽然 Q-learning 的**最终策略更优**,但它**训练时的在线回报反而更差**(−63 vs SARSA 的 −22)!因为训练中 ε-探索会让它在崖边频频掉下去。这提醒我们:**"学到的策略好" ≠ "训练过程中表现好"**——如果你在真实系统里边学边用(在线),SARSA 的保守可能更可取;若能离线学好再部署,Q-learning 的最优更香。
4. **过估计偏差**:$\max$ 会系统性**高估** Q 值(总是挑最大的、包含了正向噪声)。这是 Double Q-learning / Double DQN 要修的经典问题。

**English**:
1. **Q-learning solved Taxi** — a multi-step planning task ("pick up first, then deliver"), with 100% greedy success and near-optimal return (~+7.9). From scratch (all-zero Q-table), with only sparse −1/+20/−10 rewards and trial-and-error, it learned the entire pick-up-and-deliver routine.
2. **The cliff showdown reveals the essence of off-policy**: with identical exploration, Q-learning learns the **cliff-hugging shortest path (13 steps, optimal)**, SARSA the **cliff-avoiding safe path (17 steps)**. Because Q-learning uses $\max$ — it evaluates "what if I act greedily (no exploration) from now on," so it dares the edge; SARSA evaluates "what happens to my policy that randomizes 10% of the time," which would eventually fall off at the edge, so it detours.
3. **A counterintuitive honest result**: although Q-learning's **final policy is better**, its **online reward during training is actually worse** (−63 vs SARSA's −22)! Because ε-exploration keeps knocking it off the cliff during training. Lesson: **"learned a good policy" ≠ "performed well while learning"** — if you learn-and-act in a live system (online), SARSA's caution may be preferable; if you can learn offline then deploy, Q-learning's optimum is sweeter.
4. **Overestimation bias**: the $\max$ systematically **overestimates** Q (always picking the largest, including positive noise). This is the classic problem fixed by Double Q-learning / Double DQN.

> 💼 **实战视角 / Practical angle**
> **中文**:Q-learning 是**表格型 RL 的巅峰**、也是深度 RL 的起点。它的 off-policy 特性是**经验回放(experience replay)** 的前提——可以把历史转移存起来反复训练(→ DQN, 下节)。真实应用:游戏 AI、机器人、推荐(把用户序列当 episode)、库存/调度。**表格 Q-learning 的死穴**:状态空间一大(如围棋、像素游戏),Q 表存不下也填不满——必须用**神经网络近似 Q**,这就是 **DQN**。面试金句:*"Q-learning 用 max 学最优策略(off-policy), 所以能复用任意数据、能配经验回放; 但 max 带来过估计, 且表格法撑不起大状态空间——于是有了 DQN 和 Double DQN。"*
> **English**: Q-learning is the **peak of tabular RL** and the launchpad of deep RL. Its off-policy nature is the prerequisite for **experience replay** — store historical transitions and train on them repeatedly (→ DQN, next). Applications: game AI, robotics, recommendation (user sequences as episodes), inventory/scheduling. **Tabular Q-learning's Achilles' heel**: with a large state space (Go, pixel games), the Q-table can't be stored or filled — you must **approximate Q with a neural network**, which is **DQN**. Interview line: *"Q-learning uses max to learn the optimal policy (off-policy), so it can reuse arbitrary data and pair with experience replay; but max causes overestimation, and tables can't scale to large state spaces — hence DQN and Double DQN."*

---
### 小结 / Summary
- **中文**:Q-learning=off-policy TD 控制, 更新用 $\max_{a'}Q(s',a')$; 收敛到最优 $Q^*$, 能复用任意数据。
- **English**: Q-learning = off-policy TD control, updating with $\max_{a'}Q(s',a')$; converges to optimal $Q^*$, can reuse arbitrary data.
- **中文**:从零解出 Taxi(100%成功); 悬崖对决:Q-learning 学最优贴崖路, SARSA 学安全绕行路。
- **English**: Solves Taxi from scratch (100% success); cliff showdown: Q-learning learns the optimal cliff-edge path, SARSA the safe detour.
- **中文**:最优策略 ≠ 训练在线表现最好; max 有过估计偏差(→Double); 表格撑不起大状态空间(→DQN)。
- **English**: The optimal policy ≠ best online performance while training; max has overestimation bias (→ Double); tables can't scale (→ DQN).
